In [6]:
!pip install scipy
import numpy as np
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
from scipy.signal import convolve2d

#setting des variables de départ
image_pil = None
image_tk = None
image_originale = None
matrice_pixels = None

#création de la fenetre et du canva servant d'interface 
fenetre=tk.Tk()
fenetre.title("Application de filtres")
fenetre.geometry("800x700")
tableau=tk.Canvas(fenetre,width=600,height=400,bg="white")
tableau.pack()

#création du bouton pour charger l'image et celui pour le supp

def rafraichir():
    global image_tk
    if image_pil:
        image_tk = ImageTk.PhotoImage(image_pil)
        tableau.delete("all")
        tableau.config(width=600, height=400)
        tableau.create_image(0,0,anchor=tk.NW,image=image_tk)
cadre_boutons=tk.Frame(fenetre)
cadre_boutons.pack()

def charger_fichier():
    global image_pil,image_originale,matrice_pixels
    chemin_image=filedialog.askopenfilename(title="Image à charger")
    if chemin_image:
        image_pil = Image.open(chemin_image).convert("RGB")
        image_pil = image_pil.resize((600,400))
        image_originale = image_pil.copy()
        matrice_pixels = np.array(image_pil)
        rafraichir()

bouton_fichier=tk.Button(cadre_boutons,text="Charger une image",bg="lightblue",font=("Times",12),command=charger_fichier)
bouton_fichier.pack(side="left")

def supprimer_fichier():
    global image_pil,image_originale,image_tk
    tableau.delete("all")
    image_pil = None
    image_originale = None
    image_tk = None

bouton_supprimer=tk.Button(cadre_boutons,text="Supprimer le fichier",bg="lightblue",font=("Times",12),command=supprimer_fichier)
bouton_supprimer.pack(side="left",padx=5)

#création de la partie de manipulation d'images

#yasmine fais les boutons pour enlever le filtre precedent et le remettre que tu vas mettre dans "cadre_boutons"

txt_filtres=tk.Label(fenetre,text="ᗢ-----------Filtres-----------ᗢ")
txt_filtres.pack()

cadre_filtres=tk.Frame(fenetre)
cadre_filtres.pack()

def valider(): 
    filtre=menu.get()
    if filtre=="SEPIA":
        appliquer_sepia()
    elif filtre=="LUMINOSITE":
        appliquer_lumi()
    elif filtre=="CONTRASTE":
        appliquer_contraste()
    elif filtre=="FLOU":
        appliquer_flou()
    elif filtre=="FLOU GAUSSIEN":
        appliquer_Gauss()
    elif filtre=="NETTETE":
        appliquer_nettete()
    elif filtre=="FUSIONNER":
        fusionner()
    elif filtre=="MIROIR":
        appliquer_miroir()
    elif filtre=="INVERSE":
        appliquer_inverse()
    elif filtre=="NOIR ET BLANC":
        appliquer_noirblanc()

cadre_menu=tk.Frame(fenetre)
cadre_menu.pack()

txt_longueur=tk.Label(cadre_menu,text="Choisissez le filtre sur le menu déroulant:",font=("Times",14))
txt_longueur.pack()

menu=tk.StringVar(cadre_menu)
menu.set("SEPIA")
options=["SEPIA","LUMINOSITE","CONTRASTE","FLOU","FLOU GAUSSIEN","NETTETE","FUSIONNER","MIROIR","INVERSE","NOIR ET BLANC"]
menu_deroulant=tk.OptionMenu(cadre_menu,menu,*options)
menu_deroulant.pack(pady=10)

bouton_valider=tk.Button(cadre_menu,text="Valider",font=("Times",11),bg="#7B99FE",command=valider)
bouton_valider.pack(pady=10)

def enlever_filtre():
    global image_pil
    if image_originale:
        image_pil = image_originale.copy()
        rafraichir()

bouton_enlever=tk.Button(cadre_menu,text="Enlever le filtre",font=("Times",11),bg="#7B99FE",command=enlever_filtre)
bouton_enlever.pack(pady=10)

#filtres

def appliquer_sepia():
    global image_pil, matrice_pixels
    matrice_pixels = np.array(image_pil).astype(float)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= r+0.5*v+0.1*b
    v_prime= 0.5*r+0.8*v+0.1*b
    b_prime= 0.2*r+0.3*v+0.5*b
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    matrice_pixels[:,:,[0]]=r_prime
    matrice_pixels[:,:,[1]]=v_prime
    matrice_pixels[:,:,[2]]=b_prime
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()

def appliquer_noirblanc():
    global matrice_pixels, image_pil
    matrice_pixels = np.array(image_pil)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= 0.299*r
    v_prime= 0.587*v
    b_prime= 0.114*b
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    Y= r_prime+v_prime+b_prime
    matrice_pixels[:,:,[0,1,2]]=Y
    image_pil=Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()
    

def appliquer_lumi():
    global matrice_pixels, image_pil
    matrice_pixels = np.array(image_pil).astype(float)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= r+50
    v_prime= v+50
    b_prime= b+50
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    matrice_pixels[:,:,[0]]=r_prime
    matrice_pixels[:,:,[1]]=v_prime
    matrice_pixels[:,:,[2]]=b_prime
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()


def appliquer_contraste(): #soit se rapproche de effet poussière ou effet très lumineux selon valeur_contraste
    global matrice_pixels, image_pil
    matrice_pixels=np.array(image_pil).astype(float)
    valeur_contraste=0.15
    r=matrice_pixels[:,:,[0]]
    v=matrice_pixels[:,:,[1]]
    b=matrice_pixels[:,:,[2]]
    r_prime= (r-128)*valeur_contraste+128
    v_prime= (v-128)*valeur_contraste+128
    b_prime= (b-128)*valeur_contraste+128
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    matrice_pixels[:,:,[0]]=r_prime
    matrice_pixels[:,:,[1]]=v_prime
    matrice_pixels[:,:,[2]]=b_prime
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()

def appliquer_miroir():
    global image_pil
    global matrice_pixels
    matrice_pixels = np.array(image_pil)
    matrice_pixels= matrice_pixels[:,::-1,:] #inverse les colonnes avec le slicing
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()
    


noyau_fort= np.array([
    [0,-1,0],
    [-1,5,-1],
    [0,-1,0]
    ])

def appliquer_nettete(): #principe de convolution mais avec un noyau avec des chiffres négatifs et positifs pour accentuer les différnces de lignes au lieu de lisser l'image et l'objet comme pour flou
    global matrice_pixels, image_pil
    float_matrice=np.array(image_pil).astype(float)
    new_matrice = np.zeros_like(float_matrice)
    for i  in range (3): #convolution sur les 3 canaux; boundary = symm => évite bords noirs
        new_matrice[:,:,i] = convolve2d(float_matrice[:,:,i],noyau_fort, mode="same",boundary="symm")
    matrice_pixels=np.clip(new_matrice, 0, 255).astype(np.uint8)
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8)) #si on charge un nouveau filtre après, l'image sera MAJ et on appliquera pas le deuxieme filtre sur le flou
    rafraichir()

noyau = np.array([
    [1,  4,  6,  4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4],
    [1,  4,  6,  4, 1]
    ]) / 256

def appliquer_flou():
    global noyau, matrice_pixels, image_pil
    matrice= np.array(image_pil).astype(float)
    kernel=noyau/noyau.sum()
    new_matrice = np.zeros_like(matrice,dtype=float)
    for i  in range (3):
        new_matrice[:,:,i] = convolve2d(matrice[:,:,i],kernel, mode="same",boundary="symm")
    matrice_pixels=np.clip(new_matrice, 0, 255).astype(np.uint8)
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()


noyau = np.array([
    [1,  4,  6,  4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, 36, 24, 6],
    [4, 16, 24, 16, 4],
    [1,  4,  6,  4, 1]
    ]) / 256
# Algorithmes de filtre

def appliquer_Gauss():
    global matrice_pixels,noyau,image_pil
    #matrice du même type en float pour ne pas avoir d'arrondis et de la même taille que matrice_pixels
    float_matrice=np.array(image_pil).astype(float)
    new_matrice = np.zeros_like(float_matrice)
    for i  in range (3): #convolution sur les 3 canaux; boundary = symm => évite bords noirs
        new_matrice[:,:,i] = convolve2d(float_matrice[:,:,i],noyau, mode="same",boundary="symm")
    matrice_pixels=np.clip(new_matrice, 0, 255).astype(np.uint8)
    image_pil = Image.fromarray(matrice_pixels.astype(np.uint8)) #si on charge un nouveau filtre après, l'image sera MAJ et on appliquera pas le deuxieme filtre sur le flou
    rafraichir()

def appliquer_inverse(): #prend un pixel et donne sont inverse sur le spectre 
    global matrice_pixels, image_pil
    matrice_pixels = np.array(image_pil).astype(float)
    r= matrice_pixels[:,:,[0]]
    v= matrice_pixels[:,:,[1]]
    b= matrice_pixels[:,:,[2]]
    r_prime= 255-r
    v_prime= 255-v
    b_prime= 255-b
    r_prime = np.clip(r_prime, 0, 255)
    v_prime = np.clip(v_prime, 0, 255)
    b_prime = np.clip(b_prime, 0, 255)
    matrice_pixels[:,:,[0]]=r_prime
    matrice_pixels[:,:,[1]]=v_prime
    matrice_pixels[:,:,[2]]=b_prime
    image_pil=Image.fromarray(matrice_pixels.astype(np.uint8))
    rafraichir()

def fusionner():
    global image_pil
    if image_pil:
        chemin2 = filedialog.askopenfilename(title="Image à fusionner")
        if chemin2:
            img2 = Image.open(chemin2).convert("RGB").resize((600, 300))
            matrice1 = np.array(image_pil).astype(float)
            matrice2 = np.array(img2).astype(float)
            fusion = (matrice1 + matrice2) / 2
            image_pil = Image.fromarray(np.uint8(fusion))
            rafraichir()

#yasmine ajoute tous tes filtres et n'oublie pas de les incrémenter au menu déroulant

tableau.mainloop()


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
